<a href="https://colab.research.google.com/github/devgerlancsilva/GoogleBuildwithAI/blob/main/Iniciando_o_Gemini_TTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Gemini Text-to-speech
<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_TTS.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The Gemini API can transform text input into single speaker or multi-speaker audio podcast-like experience like in [NotebookLM](https://notebooklm.google.com/). This notebook provides an example of how to control the *Text-to-speech* (TTS) capability of the Gemini model and guide its style, accent, pace, and tone.

Before diving in the code, you should try this capability on [AI Studio](https://aistudio.google.com/prompts/new_chat?model=gemini-3.1-flash-tts-preview).

**Note that the TTS model can only do TTS, it does not have the reasoning capabilities of the Gemini models, so you can ask things like "say this in that style", but not "tell me why the sky is blue".** If that's what you want, you should use the [Live API](./Get_started_LiveAPI.ipynb) instead.

The [documentation](https://ai.google.dev/gemini-api/docs/speech-generation) is also a good place to start discovering the TTS capability.

## Setup

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

### Install and initialize the SDK

In [ ]:
!pip install -U -q "google-genai>=1.73.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 21.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.6.0 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.6.0 which is incompatible.


In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

### Select a model

Audio-out is only supported by the "`tts`" models like `gemini-3.1-flash-tts-preview`.

For more information about all Gemini models, check the [documentation](https://ai.google.dev/gemini-api/docs/models/gemini) for extended information on each of them.

In [ ]:
MODEL_ID = "gemini-3.1-flash-tts-preview" # @param ["gemini-3.1-flash-tts-preview"] {"allow-input":true, isTemplate: true}

Next create a helper function to prompt the model and play back the audio in the notebook:

In [ ]:
# @title Helper functions (just run that cell)

import contextlib
import wave
from IPython.display import Audio

file_index = 0

@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        yield wf

def play_audio_blob(blob):
  global file_index
  file_index += 1

  fname = f'audio_{file_index}.wav'
  with wave_file(fname) as wav:
    wav.writeframes(blob.data)

  return Audio(fname, autoplay=True)

def play_audio(response):
    return play_audio_blob(response.candidates[0].content.parts[0].inline_data)

## Generate a simple audio output

Let's start with something simple:

In [ ]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents="Say 'Olá, meu nome é GeminAI!'",
  config={
    "response_modalities": ["AUDIO"],
    "speech_config": types.SpeechConfig(
      voice_config=types.VoiceConfig(
        prebuilt_voice_config=types.PrebuiltVoiceConfig(
          voice_name="Kore"
        )
      )
    )
  },
)

The generated output is in the response `inline_data` and as you can see it's indeed audio data.

In [ ]:
blob = response.candidates[0].content.parts[0].inline_data
print(blob.mime_type)

audio/l16; rate=24000; channels=1


To be able to listen to the generated audio in colab, you're going to use our helper function to write the output in a file and play it.

In [ ]:
play_audio_blob(blob)

## Choose a voice

The TTS models support 30 prebuilt voice options. You can hear all of them in [AI Studio](https://aistudio.google.com/generate-speech). Use a `SpeechConfig` with `VoiceConfig` to select a specific voice:

In [ ]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents="Diga alegremente: Olá, Estudantes do IFAL tenham um Ótima Noite!",
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      voice_config=types.VoiceConfig(
        prebuilt_voice_config=types.PrebuiltVoiceConfig(
          voice_name='Puck',
        )
      )
    ),
  )
)

play_audio(response)

## Control how the model speaks

The Gemini TTS model differentiates itself from traditional TTS by using a large language model that knows not only **what** to say, but also **how** to say it.

To unlock this capability, think of yourself as a director setting a scene for a virtual voice talent to perform. By providing nuanced instructions — a precise regional accent, specific paralinguistic features (e.g. breathiness), or pacing — you can leverage the model's context awareness to generate highly dynamic, natural and expressive audio performances.

### Prompting Strategy: Audio Profile, Scene, Director's Notes, and Audio Tags

A robust prompt ideally includes these elements:

- **Audio Profile** — Establishes a persona for the voice: character identity, name, archetype, and background.
- **Scene** — Sets the stage, describing the physical environment and the emotional "vibe".
- **Director's Notes** — Performance guidance covering style, pacing, accent, and articulation.
- **Sample Context** — Gives the model a contextual starting point so your "virtual actor" enters the scene naturally.
- **Transcript** — The actual text the model will speak.
- **Audio Tags** — Inline modifiers in square brackets (e.g. `[whispers]`, `[shouting]`) placed in the transcript to change how specific parts are delivered.

For optimal performance, the transcript and directorial prompts should align — "who is saying it" should match "what is said" and "how it is being said."

Here's a full example:

In [ ]:
prompt = """
# AUDIO PROFILE: Jaz R.
## "A euforia da manhã"

São 22h em um estúdio com paredes de vidro com vista para o horizonte londrino iluminado pelo luar,
mas lá dentro, a luminosidade é ofuscante. A luz vermelha indicadora de "NO AR" está acesa.

Jaz está de pé, não sentada, saltitando na ponta dos calcanhares ao
ritmo de uma batida pulsante.

### NOTAS DO DIRETOR

Estilo:
* O "Sorriso Vocal": Você precisa ouvir o sorriso no áudio. O palato mole está
sempre elevado para manter o tom alegre, ensolarado e explicitamente convidativo.

* Dinâmica: Projeção alta sem gritar. Consoantes marcantes e vogais alongadas
em palavras que expressam entusiasmo (por exemplo, "Bom dia").

Ritmo:
Fala em um ritmo energético, acompanhando a música rápida.

Uma cadência "vibrante". Fala rápida com transições fluidas.

Sotaque:
Jaz é de Brixton, Londres.

### EXEMPLO DE CONTEXTO
Jaz é o padrão da indústria para rádio Top 40, promos de eventos eletrizantes ou
qualquer roteiro que exija um sotaque carismático do Estuário do Tâmisa e energia contagiante.

#### TRANSCRIÇÃO
[animada] Sim, a vibe no estúdio está incrível! Vocês estão conectados e
está bombando em Londres agora. Se você está preso no metrô, ou
só sentado aí fingindo que está trabalhando... pare com isso. Sério, eu estou vendo você.

[gritando] Aumenta o som! Temos o roteiro do projeto chegando em três,
dois... vamos lá!
"""

response = client.models.generate_content(
  model=MODEL_ID,
  contents=prompt,
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      voice_config=types.VoiceConfig(
        prebuilt_voice_config=types.PrebuiltVoiceConfig(
          voice_name='Kore',
        )
      )
    ),
  )
)

play_audio(response)

### Tips for Director's Notes

You don't need to include every element — sometimes giving the model space to fill in the gaps helps naturalness (just like a talented actor). Define only what's important to the performance, being careful not to overspecify. Too many strict rules will limit the model's creativity and may result in a worse performance.

**Style** — Sets the tone of the generated speech. Be descriptive: *"Infectious enthusiasm. The listener should feel like they are part of a massive, exciting community event."* works better than simply saying *"energetic and enthusiastic"*. You can even try voiceover industry terms like "vocal smile". Layer as many style characteristics as you want.

**Accent** — The more specific you are, the better. Use *"British English accent as heard in Croydon, England"* rather than *"British accent"*.

**Pacing** — Control overall speed and variation. Examples range from simple (*"Speak as fast as possible"*) to complex (*"The Drift: The tempo is incredibly slow and liquid. Words bleed into each other. There is zero urgency."*).

### Audio tags

You can change how words, sentences, or sections of your transcript are delivered by using **audio tags** — words in square brackets that indicate how something should be said, a change of tone, or an interjection.

There is no exhaustive list of what tags work. Experiment with different emotions and expressions to see how the output changes. If your transcript is not in English, for best results use English audio tags.

Here are some commonly used tags to get started:

| | | | |
|---|---|---|---|
| `[amazed]` | `[crying]` | `[curious]` | `[excited]` |
| `[sighs]` | `[gasp]` | `[giggles]` | `[laughs]` |
| `[mischievously]` | `[panicked]` | `[sarcastic]` | `[serious]` |
| `[shouting]` | `[tired]` | `[trembling]` | `[whispers]` |

Here's an example that uses audio tags directly in the transcript:

In [ ]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents='[sussurros] Pelo o Amor de Deus... [suspiro] Algo ruim está se aproximando.[shouting] Corra!',
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"], # Corrected parameter name
    speech_config=types.SpeechConfig(
      voice_config=types.VoiceConfig(
        prebuilt_voice_config=types.PrebuiltVoiceConfig(
          voice_name='Enceladus',
        )
      )
    ),
  )
)

play_audio(response)

### Simple style control

You can also control style with simpler prompts or Director's Notes alone. Here are a few examples:

In [ ]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents='Diga em um sussurro arrepiante: "Pelo Amor de Deus... Algo ruim está por vir."',
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      voice_config=types.VoiceConfig(
        prebuilt_voice_config=types.PrebuiltVoiceConfig(
          voice_name='Enceladus',
        )
      )
    ),
  )
)

play_audio(response)

Try using a voice option that corresponds to the style or emotion you want to convey, to emphasize it even more. For example, Enceladus's breathiness might emphasize "tired" and "bored", while Puck's upbeat tone could complement "excited" and "happy".

## Multi-speaker TTS

For multi-speaker audio, use a `MultiSpeakerVoiceConfig` with each speaker (up to 2) configured as a `SpeakerVoiceConfig`. The speaker names in the config **must match** the names used in your prompt:

In [ ]:
prompt = """Transcreva a seguinte conversa entre Joe e Jane usando o TTS:
Joe: Como vai você hoje, Jane?
Jane: Nada mal, e você?"""

response = client.models.generate_content(
  model=MODEL_ID,
  contents=prompt,
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
        speaker_voice_configs=[
          types.SpeakerVoiceConfig(
            speaker='Joe',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Kore',
              )
            )
          ),
          types.SpeakerVoiceConfig(
            speaker='Jane',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Puck',
              )
            )
          ),
        ]
      )
    )
  )
)

play_audio(response)

You can also provide individual style guidance for each speaker in a multi-speaker prompt:

In [ ]:
prompt = """Faça o Orador 1 parecer cansado e entediado, e o Orador 2 parecer animado e feliz:
Orador 1: Então... qual é a nossa programação para hoje?
Orador 2: Você nunca vai adivinhar!"""

response = client.models.generate_content(
  model=MODEL_ID,
  contents=prompt,
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
        speaker_voice_configs=[
          types.SpeakerVoiceConfig(
            speaker='Speaker1',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Enceladus',
              )
            )
          ),
          types.SpeakerVoiceConfig(
            speaker='Speaker2',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Puck',
              )
            )
          ),
        ]
      )
    )
  )
)

play_audio(response)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-3.1-flash-tts\nPlease retry in 51.973883002s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.1-flas

## Generate a transcript, then convert to audio

The TTS models only output audio, but you can use a regular Gemini model to generate a transcript first, then pass it to the TTS model. This is useful for creating podcast-style content:

In [ ]:
# Step 1: Generate a transcript with a regular Gemini model
transcript = client.models.generate_content(
  model="gemini-2.5-flash",
  contents="""Gere uma transcrição curta de cerca de 100 palavras que diga algo como:
Foi extraído de um podcast de herpetólogos entusiasmados.
Os nomes dos apresentadores são Dra. Anya e Liam."""
).text

print("Generated transcript:")
print(transcript)

Generated transcript:
Foi extraído de um podcast de herpetólogos entusiasmados.

---

**[Início da Transcrição]**

**Liam:** ...e é isso que me fascina, Dra. Anya. As adaptações sensoriais! As pessoas pensam que cobras são apenas visuais, mas há muito mais acontecendo, certo?

**Dra. Anya:** Absolutamente, Liam! É um erro comum. Elas são verdadeiros mestres em decodificar o ambiente. Pense na língua bífida, constantemente "provando" o ar para captar cheiros. É como ter um nariz 3D!

**Liam:** Um nariz 3D! Adoro isso. E as fossetas termorreceptoras em algumas, como as víboras? Aquilo é pura ficção científica.

**Dra. Anya:** Não é?! Detetores de calor infravermelho! Elas podem "ver" o calor de uma presa no escuro total. É uma vantagem evolutiva tão espetacular.

**Liam:** Espetacular mesmo! Realmente te faz apreciar o quão complexos e subestimados esses animais são. Nossa paixão pela herpetologia só cresce.

**Dra. Anya:** Exatamente! Cada descoberta é um pequeno milagre.

**[Fim da Tra

In [ ]:
# Step 2: Convert the transcript to audio with the TTS model
response = client.models.generate_content(
  model=MODEL_ID,
  contents=transcript,
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
        speaker_voice_configs=[
          types.SpeakerVoiceConfig(
            speaker='Dr. Anya',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Kore',
              )
            )
          ),
          types.SpeakerVoiceConfig(
            speaker='Liam',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Puck',
              )
            )
          ),
        ]
      )
    )
  )
)

play_audio(response)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-3.1-flash-tts\nPlease retry in 6.140458982s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.1-flash

## Further reading

- [Speech generation documentation](https://ai.google.dev/gemini-api/docs/speech-generation)
- [Voice Library in AI Studio](https://aistudio.google.com/apps/bundled/voice-library?showPreview=true)
- [Live API](https://ai.google.dev/gemini-api/docs/live) for interactive, real-time audio generation
- [Audio understanding](https://ai.google.dev/gemini-api/docs/audio) for working with audio inputs